### CArga de bases


In [1]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *
from variables_inicio import *
from sqlalchemy import create_engine
from sqlalchemy import text

spark = SparkSession.builder \
    .appName("SparkExample") \
    .master("local[*]") \
    .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()



In [4]:
append_table_SQL(spark,df_base,'negociosso_borrar',server_kishin,user_kishin,pwd_kishin,'DANTALION')


In [2]:
# df_retiro = df_retiro.withColumn(
#     "NUMDOCUMENTO",
#     F.right(
#         F.concat(F.lit("00000000"), F.col("NUMDOCUMENTO")),
#         F.lit(8)
#     )
# )

In [ ]:
df_retiro

In [5]:
filename='BASE_EN_NUEVOS_TARGET_202608.csv'
df_base=cargar_archivo_csv(spark,filename,'|',True)

In [3]:
query = f"""
    SELECT * FROM DANTALION.[dbo].Base_Maestra_Efectiva_Negocios_Vigente
    """
df_formato=obtener_tabla_sql(spark,query,server_kishin,user_kishin,pwd_kishin,db_kishin)


In [ ]:
fecha_mes_base='2026-08-01'

fecha_envio_base='2026-08-01'
filename='basenegociosjulio (1).csv'

df_base=cargar_archivo_csv(spark,filename,';',True)

filename='Copia de LISTA DE EXCEPCION EN TARGET - JULIO 26.csv'
df_retiro=cargar_archivo_csv(spark,filename,';',True)

In [6]:
df_base = df_retiro.withColumn(
    "NUMDOCUMENTO",
    F.right(
        F.concat(F.lit("00000000"), F.col("NUMDOCUMENTO")),
        F.lit(8)
    )
)
# df_retiro = df_retiro.withColumn(
#     "NUMDOCUMENTO",
#     F.right(
#         F.concat(F.lit("00000000"), F.col("NUMDOCUMENTO")),
#         F.lit(8)
#     )
# )
# df_retiro=df_retiro.withColumn('RETIRO',F.lit('RETIRO_EXCEPCION'))

In [7]:
df_base=df_base.dropDuplicates(['NUMDOCUMENTO'])
# df_retiro=df_retiro.dropDuplicates(['NUMDOCUMENTO'])
# df_base=df_base.join(df_retiro,['NUMDOCUMENTO'],'left')

In [8]:
print(df_base.columns)
print(df_formato.columns)

['NOMCOMERCIAL', 'SEGMICROPEQUENACOMERCIAL', 'IMPDHM', 'TIPOBASEMICROPEQUENA', 'NUMDOCUMENTO', 'NOMBRES', 'PATERNO', 'MATERNO', 'NACIMIENTO', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'CELULAR1', 'CELULAR2', 'CELULAR3', 'CELULAR4', 'CELULAR5', 'CELULAR6', 'CELULAR7', 'CELULAR8', 'CELULAR9', 'CELULAR10', 'CELULAR11', 'CELULAR12', 'CELULAR13', 'CELULAR14', 'CELULAR15', 'AREAEFECTINEGOCIOS', 'FLAG_RECURRENCIA_EFECTINEGOCIO', 'NUM_CUOTAS_PAGADAS', 'NUM_CUOTAS_IMPAGAS', 'RUC', 'EMPRESA1', 'SALDO1', 'EMPRESA2', 'SALDO2', 'EMPRESA3', 'SALDO3', 'CANAL_ASIGNADO', 'DEUDAEFECTINEGOCIO', 'DEUDAEFECTIVO', 'DEUDAELECTRO', 'DEUDAMOTOS', 'DEUDAMYPESINEFE', 'SUMASALDOVIGENTETOTAL', 'CODINDICADORMYPERU', 'CODTIPOCLIENTEEFECTINEGOCIOS', 'PROCESOS', 'ZONA', 'MONTOREFERENCIALFT', 'MONTOREFERENCIALHS', 'FLAG_FONDOCRECER']
['NUMDOCUMENTO', 'SEGMICROPEQUENACOMERCIAL', 'IMPDHM', 'TIPOBASEMICROPEQUENA', 'Nombres', 'Paterno', 'Materno', 'Nacimiento', 'Direccion', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'CELULAR1

In [9]:
exprs = [
    F.count(
        F.when(
            F.col(c).isNotNull() &
            (F.trim(F.col(c).cast("string")) != "") &
            (F.upper(F.trim(F.col(c).cast("string"))) != "NULL"),
            c
        )
    ).alias(c)
    for c in df_base.columns
]

df_counts = df_base.agg(*exprs).collect()[0].asDict()

cols_con_data = [c for c, v in df_counts.items() if v > 0]

df_base = df_base.select(cols_con_data)

In [19]:
cols_base = set(df_base.columns)
cols_formato = set(df_formato.columns)

solo_en_base = cols_base - cols_formato
print("Solo en df_base:", solo_en_base)
solo_en_formato = cols_formato -cols_base
print("Solo en formato:", solo_en_formato)


Solo en df_base: set()
Solo en formato: {'TELF2', 'RETIRO', 'MES_DURACION_BASE', 'CEL11', 'REP2', 'CEL15', 'CEL09', 'CEL12', 'recompra', 'Montos_Referenciales', 'prioridad', 'FLAT1', 'CEL13', 'ULTIMOMONTODESEMBOLSADO', 'CEL10', 'CEL02', 'FLAT3', 'CEL04', 'TELF5', 'Direccion', 'REP1', 'CEL05', 'REP3', 'TELF8', 'TELF7', 'TELF4', 'num_cuotas_impagas', 'DEVUELTO', 'num_cuotas_pagadas', 'FLAT2', 'FECHA_ENVIO', 'CEL06', 'CEL14', 'TELF1', 'PERFIL_IC', 'CEL03', 'CEL01', 'REP4', 'PERFILINICIAL', 'AÑO_DURACION_BASE', 'ULTIMOMONTODESEMBOLSADOPLUS20', 'CEL08', 'CEL07', 'TELF3', 'flag_recurrencia_efectinegocio', 'SERVICIO', 'TELF6', 'flag_consentimiento'}


In [18]:
df_base=df_base.drop(
'FLAG_RECURRENCIA_EFECTINEGOCIO', 'NOMCOMERCIAL', 'DEUDAEFECTINEGOCIO', 'CODTIPOCLIENTEEFECTINEGOCIOS'
)

In [14]:
df_base=df_base.withColumnRenamed('PATERNO','Paterno')
df_base=df_base.withColumnRenamed('PATERNO','Paterno')
df_base=df_base.withColumnRenamed('PROCESOS','Proceso')
df_base=df_base.withColumnRenamed('RUC','ruc')
df_base=df_base.withColumnRenamed('EMPRESA1','empresa1')
df_base=df_base.withColumnRenamed('MATERNO','Materno')
df_base=df_base.withColumnRenamed('EMPRESA2','empresa2')
df_base=df_base.withColumnRenamed('EMPRESA3','empresa3')
df_base=df_base.withColumnRenamed('SALDO1','RangoSaldo1')
df_base=df_base.withColumnRenamed('SALDO2','RangoSaldo2')
df_base=df_base.withColumnRenamed('SALDO3','RangoSaldo3')
df_base=df_base.withColumnRenamed('NOMBRES','Nombres')
df_base=df_base.withColumnRenamed('ZONA','Zona')
df_base=df_base.withColumnRenamed('NACIMIENTO','Nacimiento')


In [ ]:
# df_base=df_base.withColumnRenamed('DEUDAMYPESINEFE','Montos_Referenciales')
# # df_base=df_base.withColumnRenamed('DEUDAELECTRO','MONTOREFERENCIALFT')
# df_base=df_base.withColumnRenamed('DEUDAMOTOS','MONTOREFERENCIALHS')
# df_base=df_base.drop('DEUDAEFECTIVO','DEUDAELECTRO')
# df_base=df_base.withColumnRenamed('DEUDAEFECTIVO','pendiente')


In [20]:
df_base.show(5)

+------------------------+---------+--------------------+------------+---------------+----------+----------+--------------------+------------+---------+--------------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+------------------+-----------+--------------------+-----------+--------+-----------+--------+-----------+--------------+-------------+------------+----------+---------------+-------+----+------------------+------------------+
|SEGMICROPEQUENACOMERCIAL|   IMPDHM|TIPOBASEMICROPEQUENA|NUMDOCUMENTO|        Nombres|   Paterno|   Materno|          Nacimiento|DEPARTAMENTO|PROVINCIA|      DISTRITO| CELULAR1| CELULAR2| CELULAR3| CELULAR4| CELULAR5| CELULAR6| CELULAR7| CELULAR8| CELULAR9|CELULAR10|CELULAR11|CELULAR12|CELULAR13|CELULAR14|CELULAR15|AREAEFECTINEGOCIOS|        ruc|            empresa1|RangoSaldo1|empresa2|RangoSaldo2|empresa3|RangoSaldo3|CANAL_ASIGNADO|DEUDAEFECTIVO|DEUD

In [ ]:
['NUMDOCUMENTO', 'SEGMICROPEQUENACOMERCIAL', 'IMPDHM', 'TIPOBASEMICROPEQUENA', 'Nombres', 'Paterno', 'Materno', 'Nacimiento', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'AREAEFECTINEGOCIOS', 'ruc', 'empresa1', 'RangoSaldo1', 'empresa2', 'RangoSaldo2', 'empresa3', 'RangoSaldo3', 'CANAL_ASIGNADO', 'DEUDAEFECTIVO', 'DEUDAELECTRO', 'DEUDAMOTOS', 'Proceso', 'Zona', 'MONTOREFERENCIALFT', 'MONTOREFERENCIALHS', 'RETIRO', 'CEL01', 'CEL02', 'CEL03', 'CEL04', 'CEL05', 'CEL06', 'CEL07', 'CEL08', 'CEL09', 'CEL10', 'CEL11', 'CEL12', 'CEL13', 'CEL14', 'CEL15', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'FECHA_ENVIO', 'SERVICIO']

['NUMDOCUMENTO', 'SEGMICROPEQUENACOMERCIAL', 'IMPDHM', 'TIPOBASEMICROPEQUENA', 'Nombres', 'Paterno', 'Materno', 'Nacimiento', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'AREAEFECTINEGOCIOS', 'ruc', 'empresa1', 'RangoSaldo1', 'empresa2', 'RangoSaldo2', 'empresa3', 'RangoSaldo3', 'CANAL_ASIGNADO', 'DEUDAEFECTIVO', 'DEUDAELECTRO', 'DEUDAMOTOS', 'Proceso', 'Zona', 'MONTOREFERENCIALFT', 'MONTOREFERENCIALHS', 'RETIRO', 'CEL01', 'CEL02', 'CEL03', 'CEL04', 'CEL05', 'CEL06', 'CEL07', 'CEL08', 'CEL09', 'CEL10', 'CEL11', 'CEL12', 'CEL13', 'CEL14', 'CEL15', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'FECHA_ENVIO', 'SERVICIO']


In [49]:
df_base.select(
    'DEUDAMYPESINEFE', 'DEUDAELECTRO', 'DEUDAMOTOS', 'DEUDAEFECTIVO'
    ).distinct().show()

+---------------+------------+----------+-------------+
|DEUDAMYPESINEFE|DEUDAELECTRO|DEUDAMOTOS|DEUDAEFECTIVO|
+---------------+------------+----------+-------------+
|           0.00|     4332.41|      0.00|         0.00|
|        1000.00|        0.00|      0.00|      2144.77|
|           0.00|     1149.30|      0.00|         0.00|
|           0.00|        0.00|      0.00|     35939.10|
|       14608.01|        0.00|      0.00|       243.81|
|       29678.99|     1670.08|      0.00|         0.00|
|           0.00|     1349.30|      0.00|         0.00|
|           0.00|     5207.02|      0.00|         0.00|
|        1238.02|        0.00|      0.00|      6266.22|
|           0.00|        0.00|      0.00|      1893.05|
|           0.00|      407.39|      0.00|         0.00|
|           0.00|     1368.42|      0.00|         0.00|
|           0.00|      352.05|      0.00|      1639.95|
|        8537.92|        0.00|   4810.65|       326.04|
|           0.00|     1031.62|      0.00|       

In [41]:
df_base=df_base.drop('DEUDAEFECTINEGOCIO','CODTIPOCLIENTEEFECTINEGOCIOS','NOMCOMERCIAL','FLAG_RECURRENCIA_EFECTINEGOCIO','DEUDAMYPESINEFE','DEUDAMYPESINEFE', 'DEUDAELECTRO', 'DEUDAMOTOS', 'DEUDAEFECTIVO')

In [19]:
df_base.select('NOMCOMERCIAL').distinct().show()

+--------------------+
|        NOMCOMERCIAL|
+--------------------+
|EFECTINEGOCIO BAN...|
+--------------------+



In [42]:
df_base=df_base.withColumnRenamed('PATERNO','Paterno')
df_base=df_base.withColumnRenamed('MATERNO','Materno')
df_base=df_base.withColumnRenamed('PROCESOS','Proceso')


In [21]:
df_base=df_base.withColumn('Proceso',when(F.col('Proceso')==1,'FASTTRACK')
                                    .when(F.col('Proceso')==2,F.lit('HUMANO SEGURO'))
                                    .otherwise(F.lit('FULL'))
                                    )

In [23]:
df_base.show(5)

+------------------------+---------+--------------------+------------+---------------+----------+----------+--------------------+------------+---------+--------------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+---------+------------------+-----------+--------------------+-----------+--------+-----------+--------+-----------+--------------+-------------+------------+----------+---------------+---------+----+------------------+------------------+
|SEGMICROPEQUENACOMERCIAL|   IMPDHM|TIPOBASEMICROPEQUENA|NUMDOCUMENTO|        Nombres|   Paterno|   Materno|          Nacimiento|DEPARTAMENTO|PROVINCIA|      DISTRITO| CELULAR1| CELULAR2| CELULAR3| CELULAR4| CELULAR5| CELULAR6| CELULAR7| CELULAR8| CELULAR9|CELULAR10|CELULAR11|CELULAR12|CELULAR13|CELULAR14|CELULAR15|AREAEFECTINEGOCIOS|        ruc|            empresa1|RangoSaldo1|empresa2|RangoSaldo2|empresa3|RangoSaldo3|CANAL_ASIGNADO|DEUDAEFECTIVO|DE

In [25]:
cols_cel = [f"CELULAR{i}" for i in range(1, 16)]
cols_cel

['CELULAR1',
 'CELULAR2',
 'CELULAR3',
 'CELULAR4',
 'CELULAR5',
 'CELULAR6',
 'CELULAR7',
 'CELULAR8',
 'CELULAR9',
 'CELULAR10',
 'CELULAR11',
 'CELULAR12',
 'CELULAR13',
 'CELULAR14',
 'CELULAR15']

In [26]:
cols_cel = [f"CELULAR{i}" for i in range(1, 10)]

df_base = df_base.withColumn(
    "CELULARES_ARRAY",
    F.array(*[F.col(c).cast("string") for c in cols_cel])
)

df_base = df_base.withColumn(
    "CELULARES_VALIDOS",
    F.array_distinct(
        F.filter(
            F.transform(
                F.col("CELULARES_ARRAY"),
                lambda x: F.regexp_replace(F.trim(x), r"\D", "")
            ),
            lambda x: x.rlike(r"^9\d{8}$")
        )
    )
)

max_cel = 15

for i in range(max_cel):
    df_base = df_base.withColumn(
        f"CEL{str(i+1).zfill(2)}",
        F.expr(f"get(CELULARES_VALIDOS, {i})")
    )

df_base = df_base.drop(
    "CELULARES_ARRAY",
    "CELULARES_VALIDOS",
    *cols_cel
)

In [27]:
df_base=df_base.withColumn('AÑO_DURACION_BASE',F.lit('2026'))
df_base=df_base.withColumn('MES_DURACION_BASE',F.lit('08'))
df_base=df_base.withColumn('FECHA_ENVIO',F.lit('2026-08-01'))
df_base=df_base.withColumn('SERVICIO',F.lit('04'))


In [28]:
print(df_base.count())
print(df_base.dropDuplicates(['NUMDOCUMENTO']).count())

148486
148486


In [32]:
print(df_base.columns)

['SEGMICROPEQUENACOMERCIAL', 'IMPDHM', 'TIPOBASEMICROPEQUENA', 'NUMDOCUMENTO', 'Nombres', 'Paterno', 'Materno', 'Nacimiento', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'CELULAR10', 'AREAEFECTINEGOCIOS', 'ruc', 'empresa1', 'RangoSaldo1', 'empresa2', 'RangoSaldo2', 'RangoSaldo3', 'CANAL_ASIGNADO', 'MONTOREFERENCIALFT', 'MONTOREFERENCIALHS', 'Montos_Referenciales', 'Proceso', 'Zona', 'MONTOREFERENCIALFT', 'CEL01', 'CEL02', 'CEL03', 'CEL04', 'CEL05', 'CEL06', 'CEL07', 'CEL08', 'CEL09', 'CEL10', 'CEL11', 'CEL12', 'CEL13', 'CEL14', 'CEL15', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'FECHA_ENVIO', 'SERVICIO']


In [ ]:
['SEGMICROPEQUENACOMERCIAL', 'IMPDHM', 'TIPOBASEMICROPEQUENA', 'NUMDOCUMENTO', 'Nombres', 'Paterno', 'Materno', 'Nacimiento', 'DEPARTAMENTO', 'PROVINCIA', 'DISTRITO', 'CELULAR10', 'AREAEFECTINEGOCIOS', 'ruc', 'empresa1', 'RangoSaldo1', 'empresa2', 'RangoSaldo2', 'RangoSaldo3', 'CANAL_ASIGNADO', 'MONTOREFERENCIALFT', 'MONTOREFERENCIALHS', 'Montos_Referenciales', 'Proceso', 'Zona', 'MONTOREFERENCIALFT', 'CEL01', 'CEL02', 'CEL03', 'CEL04', 'CEL05', 'CEL06', 'CEL07', 'CEL08', 'CEL09', 'CEL10', 'CEL11', 'CEL12', 'CEL13', 'CEL14', 'CEL15', 'AÑO_DURACION_BASE', 'MES_DURACION_BASE', 'FECHA_ENVIO', 'SERVICIO']

In [29]:
df_base.show(5)

+------------------------+---------+--------------------+------------+---------------+----------+----------+--------------------+------------+---------+--------------+---------+---------+---------+---------+---------+---------+------------------+-----------+--------------------+-----------+--------+-----------+--------+-----------+--------------+-------------+------------+----------+---------------+---------+----+------------------+------------------+---------+---------+---------+---------+---------+---------+---------+---------+---------+-----+-----+-----+-----+-----+-----+-----------------+-----------------+-----------+--------+
|SEGMICROPEQUENACOMERCIAL|   IMPDHM|TIPOBASEMICROPEQUENA|NUMDOCUMENTO|        Nombres|   Paterno|   Materno|          Nacimiento|DEPARTAMENTO|PROVINCIA|      DISTRITO|CELULAR10|CELULAR11|CELULAR12|CELULAR13|CELULAR14|CELULAR15|AREAEFECTINEGOCIOS|        ruc|            empresa1|RangoSaldo1|empresa2|RangoSaldo2|empresa3|RangoSaldo3|CANAL_ASIGNADO|DEUDAEFECTIVO

In [30]:
df_base=df_base.filter(F.col('NUMDOCUMENTO').isNotNull())

In [31]:
append_table_SQL(spark,df_base,'Base_Maestra_Efectiva_Negocios',server_kishin,user_kishin,pwd_kishin,'DANTALION')


In [51]:
exec_query_sql(server_kishin, db_kishin, user_kishin, pwd_kishin, "tNumeros_Efectiva_Negocios", "SP tNumeros Consumo")
exec_query_sql(server_zeus, "ODIN", user_zeus, pwd_zeus, "EXEC Sp_Actualizar_Efectiva_Negocios", "SP actualizar Consumo Zeus")
exec_query_sql(server_sa, "ODIN", user_sa, pwd_sa, "EXEC Sp_Actualizar_Efectiva_Negocios", "SP actualizar Consumo SA")

SP tNumeros Consumo | realizado | duración: 333.21 seg
SP actualizar Consumo Zeus | realizado | duración: 97.17 seg
SP actualizar Consumo SA | realizado | duración: 10.5 seg
